In [1]:
# import melt_functions as ice_melt
import numpy as np
import xarray as xr
import scipy.io as sio
# from plot_icebergshape import plot_icebergshape
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, interp2d
from matplotlib import cm,colors
import pickle
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import pandas as pd

In [2]:
# read in volume csvs
root = '../data/csv/iceberg_volume/'

helheim_awVol = pd.read_csv(f'{root}helheim_awVol.csv', index_col=['length'])
helheim_uwVol = pd.read_csv(f'{root}helheim_uwVol.csv', index_col=['length'])

jkb_awVol = pd.read_csv(f'{root}jkb_awVol.csv', index_col=['length'])
jkb_uwVol = pd.read_csv(f'{root}jkb_uwVol.csv', index_col=['length'])

kanger_awVol = pd.read_csv(f'{root}kanger_awVol.csv', index_col=['length'])
kanger_uwVol = pd.read_csv(f'{root}kanger_uwVol.csv', index_col=['length'])

upvk_awVol = pd.read_csv(f'{root}upernavik_awVol.csv', index_col=['length'])
upvk_uwVol = pd.read_csv(f'{root}upernavik_uwVol.csv', index_col=['length'])

In [3]:
uwVol_dict = {'helheim': helheim_uwVol,
              'jkb': jkb_uwVol,
              'kanger': kanger_uwVol,
              'upvk': upvk_uwVol
            }

awVol_dict = {'helheim': helheim_awVol,
              'jkb': jkb_awVol,
              'kanger': kanger_awVol,
              'upvk': upvk_awVol
            }

In [4]:
def vol_table(vol_dict):

    sum_dict = {}
    for key, value in vol_dict.items():
        df_sum = value.sum()
        df_sum.index = np.arange(1,len(df_sum)+1)
    
        sum_dict[key] = df_sum
    
    return sum_dict

In [5]:
uwVol_dict['helheim'].sum()

2016-04-24    3.627406e+09
2018-05-04    4.165644e+09
2020-09-03    5.680725e+09
2023-07-27    8.737880e+09
2024-08-22    6.523229e+09
dtype: float64

In [6]:
hel_sum = uwVol_dict['helheim'].sum()
hel_sum.index = np.arange(1,len(hel_sum)+1)
hel_sum

1    3.627406e+09
2    4.165644e+09
3    5.680725e+09
4    8.737880e+09
5    6.523229e+09
dtype: float64

In [7]:
len(hel_sum)

5

In [8]:
uwVol_sum_dict = vol_table(uwVol_dict)
uwVol_sum_dict

{'helheim': 1    3.627406e+09
 2    4.165644e+09
 3    5.680725e+09
 4    8.737880e+09
 5    6.523229e+09
 dtype: float64,
 'jkb': 1    6.659106e+09
 2    1.004487e+10
 3    8.281943e+09
 4    1.657052e+10
 5    1.817078e+10
 dtype: float64,
 'kanger': 1    8.591887e+09
 2    8.500883e+09
 3    4.366599e+09
 4    1.038722e+10
 5    6.853419e+09
 dtype: float64,
 'upvk': 1    6.637224e+09
 2    5.706617e+09
 3    2.001348e+10
 4    1.507404e+10
 5    8.980397e+09
 dtype: float64}

In [9]:
uwVol_sum_df = pd.DataFrame.from_dict(uwVol_sum_dict, orient='index')
uwVol_sum_df_km3 = uwVol_sum_df / 1e9
uwVol_sum_df_km3

,1,2,3,4,5
helheim,3.627406,4.165644,5.680725,8.737880,6.523229
jkb,6.659106,10.044873,8.281943,16.570522,18.170779
kanger,8.591887,8.500883,4.366599,10.387220,6.853419
upvk,6.637224,5.706617,20.013480,15.074043,8.980397


In [10]:
awVol_sum_dict = vol_table(awVol_dict)
awVol_sum_dict

{'helheim': 1    1.211545e+09
 2    1.212521e+09
 3    1.322534e+09
 4    3.214060e+09
 5    2.447940e+09
 dtype: float64,
 'jkb': 1    2.477777e+09
 2    4.020536e+09
 3    1.691958e+09
 4    5.448224e+09
 5    4.487674e+09
 dtype: float64,
 'kanger': 1    2.722185e+09
 2    2.777097e+09
 3    1.274157e+09
 4    2.345602e+09
 5    2.171614e+09
 dtype: float64,
 'upvk': 1    2.184115e+09
 2    2.362743e+09
 3    5.897315e+09
 4    5.460129e+09
 5    1.208949e+09
 dtype: float64}

In [11]:
awVol_sum_df = pd.DataFrame.from_dict(awVol_sum_dict, orient='index')
awVol_sum_df_km3 = awVol_sum_df / 1e9
awVol_sum_df_km3

,1,2,3,4,5
helheim,1.211545,1.212521,1.322534,3.214060,2.447940
jkb,2.477777,4.020536,1.691958,5.448224,4.487674
kanger,2.722185,2.777097,1.274157,2.345602,2.171614
upvk,2.184115,2.362743,5.897315,5.460129,1.208949


In [12]:
vol_frac = awVol_sum_df / uwVol_sum_df
vol_frac

,1,2,3,4,5
helheim,0.333998,0.291077,0.232811,0.367831,0.375265
jkb,0.372088,0.400257,0.204295,0.328790,0.246972
kanger,0.316832,0.326683,0.291796,0.225816,0.316866
upvk,0.329071,0.414036,0.294667,0.362221,0.134621


In [13]:
# save dfs as csvs
awVol_sum_df_km3.to_csv(f'{root}awVol_sum_df_km3.csv')
uwVol_sum_df_km3.to_csv(f'{root}uwVol_sum_df_km3.csv')
vol_frac.to_csv(f'{root}vol_frac.csv')

In [14]:
vol_frac.mean(axis=None)

np.float64(0.3082996086653721)

In [15]:
uwVol_sum_df.mean(axis=None)/1e9

np.float64(9.178693842785659)

In [16]:
uwVol_sum_df.min(axis=None)/1e9

3.6274057732793636

In [17]:
uwVol_sum_df.max(axis=None)/1e9

20.013480168992857

In [18]:
uwVol_sum_df.mean(axis=1)/1e9

helheim     5.746977
jkb        11.945445
kanger      7.740002
upvk       11.282352
dtype: float64

In [19]:
uwVol_sum_df.std(axis=1)/1e9

helheim    2.033818
jkb        5.126522
kanger     2.262647
upvk       6.094558
dtype: float64

In [20]:
awVol_sum_df_km3.std(axis=1)

helheim    0.909036
jkb        1.522956
kanger     0.605808
upvk       2.111465
dtype: float64

In [21]:
awVol_sum_df_km3.mean(axis=1)

helheim    1.881720
jkb        3.625234
kanger     2.258131
upvk       3.422650
dtype: float64

In [22]:
vol_frac.mean(axis=1)

helheim    0.320196
jkb        0.310481
kanger     0.295599
upvk       0.306923
dtype: float64